<a href="https://colab.research.google.com/github/juliablaz2003/NLP/blob/main/NLP_hands_on_2_ngram_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Hands-on #2: N-gram Language Models

In this notebook we build progressively more sophisticated **statistical language models**:

1. Unigram intuition
2. Bigram language model
3. Text generation
4. Laplace smoothing
5. Trigram language model
6. Training on a **larger corpus**

The goal is to understand how classical NLP language models work internally.

## 1. Toy Dataset

We start with a small dataset so we can inspect everything easily.

In [ ]:
text = '''
I like natural language processing
I like machine learning
I like deep learning
natural language processing is fun
machine learning is powerful
deep learning is powerful
'''
print(text)


I like natural language processing
I like machine learning
I like deep learning
natural language processing is fun
machine learning is powerful
deep learning is powerful



## 2. Tokenization

We convert the corpus into a list of tokens.

In [ ]:
tokens = text.lower().split()
tokens

['i',
 'like',
 'natural',
 'language',
 'processing',
 'i',
 'like',
 'machine',
 'learning',
 'i',
 'like',
 'deep',
 'learning',
 'natural',
 'language',
 'processing',
 'is',
 'fun',
 'machine',
 'learning',
 'is',
 'powerful',
 'deep',
 'learning',
 'is',
 'powerful']

Vocabulary:

In [ ]:
vocab = sorted(set(tokens))
vocab, len(vocab)

(['deep',
  'fun',
  'i',
  'is',
  'language',
  'learning',
  'like',
  'machine',
  'natural',
  'powerful',
  'processing'],
 11)

## 3. Bigram Counts

A **bigram** is a pair of consecutive words.

Example:

`("machine", "learning")`

In [ ]:
from collections import defaultdict

bigram_counts = defaultdict(int)

for i in range(len(tokens)-1):
    bigram = (tokens[i], tokens[i+1])
    bigram_counts[bigram] += 1

bigram_counts

defaultdict(int,
            {('i', 'like'): 3,
             ('like', 'natural'): 1,
             ('natural', 'language'): 2,
             ('language', 'processing'): 2,
             ('processing', 'i'): 1,
             ('like', 'machine'): 1,
             ('machine', 'learning'): 2,
             ('learning', 'i'): 1,
             ('like', 'deep'): 1,
             ('deep', 'learning'): 2,
             ('learning', 'natural'): 1,
             ('processing', 'is'): 1,
             ('is', 'fun'): 1,
             ('fun', 'machine'): 1,
             ('learning', 'is'): 2,
             ('is', 'powerful'): 2,
             ('powerful', 'deep'): 1})

## 4. Context Counts

We count how many times each word appears as the **first element of a bigram**.

In [ ]:
context_counts = defaultdict(int)

for (w1, w2), count in bigram_counts.items():
    context_counts[w1] += count

context_counts

defaultdict(int,
            {'i': 3,
             'like': 3,
             'natural': 2,
             'language': 2,
             'processing': 2,
             'machine': 2,
             'learning': 4,
             'deep': 2,
             'is': 3,
             'fun': 1,
             'powerful': 1})

## 5. Bigram Probabilities

We estimate:

P(w₂ | w₁)

In [ ]:
bigram_probs = {}

for (w1, w2), count in bigram_counts.items():
    bigram_probs[(w1, w2)] = count / context_counts[w1]

bigram_probs

{('i', 'like'): 1.0,
 ('like', 'natural'): 0.3333333333333333,
 ('natural', 'language'): 1.0,
 ('language', 'processing'): 1.0,
 ('processing', 'i'): 0.5,
 ('like', 'machine'): 0.3333333333333333,
 ('machine', 'learning'): 1.0,
 ('learning', 'i'): 0.25,
 ('like', 'deep'): 0.3333333333333333,
 ('deep', 'learning'): 1.0,
 ('learning', 'natural'): 0.25,
 ('processing', 'is'): 0.5,
 ('is', 'fun'): 0.3333333333333333,
 ('fun', 'machine'): 1.0,
 ('learning', 'is'): 0.5,
 ('is', 'powerful'): 0.6666666666666666,
 ('powerful', 'deep'): 1.0}

## 6. Text Generation with Bigrams

In [ ]:
import random

def generate_bigram(start_word, length=12):
    word = start_word
    sentence = [word]

    for _ in range(length):

        candidates = [(w2, p) for (w1, w2), p in bigram_probs.items() if w1 == word]

        if not candidates:
            break

        words, probs = zip(*candidates)
        word = random.choices(words, probs)[0]
        sentence.append(word)

    return " ".join(sentence)

generate_bigram("i")

'i like natural language processing is fun machine learning is fun machine learning'

## 7. Zero Probability Problem

If a word pair never appears in the training corpus, its probability is **0**.

Example:

P(pizza | like) = 0

## 8. Laplace (Add‑1) Smoothing

In [ ]:
V = len(vocab)

smoothed_bigram_probs = {}

for w1 in vocab:
    for w2 in vocab:

        count = bigram_counts.get((w1, w2), 0)
        smoothed_bigram_probs[(w1, w2)] = (count + 1) / (context_counts[w1] + V)

list(smoothed_bigram_probs.items())[:10]

[(('deep', 'deep'), 0.07692307692307693),
 (('deep', 'fun'), 0.07692307692307693),
 (('deep', 'i'), 0.07692307692307693),
 (('deep', 'is'), 0.07692307692307693),
 (('deep', 'language'), 0.07692307692307693),
 (('deep', 'learning'), 0.23076923076923078),
 (('deep', 'like'), 0.07692307692307693),
 (('deep', 'machine'), 0.07692307692307693),
 (('deep', 'natural'), 0.07692307692307693),
 (('deep', 'powerful'), 0.07692307692307693)]

# Trigram Language Model

Now we extend the model so the next word depends on **two previous words**.

P(w₃ | w₁, w₂)

In [ ]:
trigram_counts = defaultdict(int)

for i in range(len(tokens)-2):
    trigram = (tokens[i], tokens[i+1], tokens[i+2])
    trigram_counts[trigram] += 1

trigram_counts

defaultdict(int,
            {('i', 'like', 'natural'): 1,
             ('like', 'natural', 'language'): 1,
             ('natural', 'language', 'processing'): 2,
             ('language', 'processing', 'i'): 1,
             ('processing', 'i', 'like'): 1,
             ('i', 'like', 'machine'): 1,
             ('like', 'machine', 'learning'): 1,
             ('machine', 'learning', 'i'): 1,
             ('learning', 'i', 'like'): 1,
             ('i', 'like', 'deep'): 1,
             ('like', 'deep', 'learning'): 1,
             ('deep', 'learning', 'natural'): 1,
             ('learning', 'natural', 'language'): 1,
             ('language', 'processing', 'is'): 1,
             ('processing', 'is', 'fun'): 1,
             ('is', 'fun', 'machine'): 1,
             ('fun', 'machine', 'learning'): 1,
             ('machine', 'learning', 'is'): 1,
             ('learning', 'is', 'powerful'): 2,
             ('is', 'powerful', 'deep'): 1,
             ('powerful', 'deep', 'learning'): 1,
  

Context counts for trigrams:

In [ ]:
trigram_context_counts = defaultdict(int)

for (w1, w2, w3), count in trigram_counts.items():
    trigram_context_counts[(w1, w2)] += count

trigram_context_counts

defaultdict(int,
            {('i', 'like'): 3,
             ('like', 'natural'): 1,
             ('natural', 'language'): 2,
             ('language', 'processing'): 2,
             ('processing', 'i'): 1,
             ('like', 'machine'): 1,
             ('machine', 'learning'): 2,
             ('learning', 'i'): 1,
             ('like', 'deep'): 1,
             ('deep', 'learning'): 2,
             ('learning', 'natural'): 1,
             ('processing', 'is'): 1,
             ('is', 'fun'): 1,
             ('fun', 'machine'): 1,
             ('learning', 'is'): 2,
             ('is', 'powerful'): 1,
             ('powerful', 'deep'): 1})

Trigram probabilities:

In [ ]:
trigram_probs = {}

for (w1, w2, w3), count in trigram_counts.items():
    trigram_probs[(w1, w2, w3)] = count / trigram_context_counts[(w1, w2)]

trigram_probs

{('i', 'like', 'natural'): 0.3333333333333333,
 ('like', 'natural', 'language'): 1.0,
 ('natural', 'language', 'processing'): 1.0,
 ('language', 'processing', 'i'): 0.5,
 ('processing', 'i', 'like'): 1.0,
 ('i', 'like', 'machine'): 0.3333333333333333,
 ('like', 'machine', 'learning'): 1.0,
 ('machine', 'learning', 'i'): 0.5,
 ('learning', 'i', 'like'): 1.0,
 ('i', 'like', 'deep'): 0.3333333333333333,
 ('like', 'deep', 'learning'): 1.0,
 ('deep', 'learning', 'natural'): 0.5,
 ('learning', 'natural', 'language'): 1.0,
 ('language', 'processing', 'is'): 0.5,
 ('processing', 'is', 'fun'): 1.0,
 ('is', 'fun', 'machine'): 1.0,
 ('fun', 'machine', 'learning'): 1.0,
 ('machine', 'learning', 'is'): 0.5,
 ('learning', 'is', 'powerful'): 1.0,
 ('is', 'powerful', 'deep'): 1.0,
 ('powerful', 'deep', 'learning'): 1.0,
 ('deep', 'learning', 'is'): 0.5}

## Trigram Text Generator

In [ ]:
def generate_trigram(start_w1, start_w2, length=12):

    w1, w2 = start_w1, start_w2
    sentence = [w1, w2]

    for _ in range(length):

        candidates = [(w3, p) for (a, b, w3), p in trigram_probs.items() if a == w1 and b == w2]

        if not candidates:
            break

        words, probs = zip(*candidates)
        w3 = random.choices(words, probs)[0]

        sentence.append(w3)
        w1, w2 = w2, w3

    return " ".join(sentence)

generate_trigram("i", "like")

'i like machine learning i like natural language processing i like deep learning is'

# Larger Corpus Example

Now let's train on a **longer text** (excerpt from *Alice in Wonderland*, public domain).

In [ ]:
long_text = '''
Nato nel quartiere di Flores a Buenos Aires il 17 dicembre 1936 da una famiglia di origini piemontesi e liguri e appartenente alla piccola borghesia, è il primogenito di Mario Bergoglio (1908-1961) e di Regina Maria Sivori (1911-1981). Dopo di lui nasceranno: Oscar Adrian (1938-1997), Marta Regina (1940-2007), Alberto Horacio (1942-2010) e Maria Elena (1948). Da parte di padre, il bisnonno Francesco era nativo di Montechiaro d'Asti,[10] mentre il nonno Giovanni Angelo era nato in località Bricco Marmorito[11] di Portacomaro Stazione, frazione di Asti non lontana da Portacomaro, ove è sopravvissuto un ramo della famiglia;[17] la nonna Rosa era originaria di Piana Crixia in provincia di Savona. Da parte materna, il nonno era originario di Santa Giulia di Centaura, frazione collinare di Lavagna in provincia di Genova; la nonna era originaria della frazione Teo di Cabella Ligure, in provincia di Alessandria.

Da giovane ebbe una fidanzata alla quale un giorno si dichiarò dicendo "Se non mi sposo con te, mi farò prete", ma lei non gli rispose mai. Studiò chimica presso una scuola tecnica argentina e ottenne un diploma di tecnico chimico,[24] si mantenne per un certo periodo facendo le pulizie in una fabbrica di calzini e poi facendo anche il buttafuori in un locale malfamato di Córdoba.

All'età di 17 anni decise di intraprendere la vocazione sacerdotale. A 22 anni entrò nel seminario diocesano di Villa Devoto, un barrio di Buenos Aires allora retto da sacerdoti gesuiti e dopo qualche tempo decise di entrare nella Compagnia di Gesù. Nel 1960 venne inviato in Cile per completare il noviziato. L'anno successivo tornò in Argentina per continuare gli studi umanistici. Studiò filosofia e ottenne la laurea in teologia presso il Colegio Máximo de San Miguel. Imparò anche il francese, l'italiano, il tedesco, l'inglese, il latino e il greco.

Il 13 dicembre 1969 fu ordinato sacerdote dall'arcivescovo di Córdoba monsignor Castellano.

Tra il 1970 e il 1971, continuò la sua formazione in Spagna e il 22 aprile 1973 prese l'impegno solenne e definitivo all'interno dell'Ordine dei Gesuiti. Tornato in Argentina, divenne maestro di novizi, professore di teologia, consultore della provincia dei gesuiti e rettore del Collegio. Il 31 luglio 1973 venne nominato provinciale dei gesuiti dell'Argentina. Dopo sei anni, tra il 1980 e il 1986, tornò a lavorare nel settore universitario come rettore del collegio di San Giuseppe e parroco a San Miguel. Nel 1986 si recò in Germania per completare la sua tesi dottorale, ma poi venne mandato a Buenos Aires e successivamente a Córdoba, dove lavorò come direttore spirituale e confessore.


L'arcivescovo Bergoglio nel 2008
Rapporto con la teologia della liberazione
La teologia della liberazione nacque negli anni '60 come una risposta alla povertà e alle ingiustizie sociali in America Latina. Questa corrente di pensiero cristiano si concentrò sulla liberazione dei poveri e degli oppressi, adottando una visione radicale della giustizia sociale che prevedeva un cambiamento nelle strutture politiche ed economiche. Alcuni esponenti del movimento, come Gustavo Gutiérrez, rifiutavano l'idea di un cristianesimo "spiritualista" e ritenevano che la Chiesa dovesse impegnarsi direttamente contro le ingiustizie, talvolta associandosi con movimenti politici di sinistra.

Il rapporto di Bergoglio con la teologia della liberazione è complesso e articolato, segnato da una posizione inizialmente distaccata e cauta, ma anche da un impegno profondo nel campo sociale e religioso in Argentina, specialmente durante gli anni della dittatura militare.

Nel contesto argentino, Bergoglio, che all'epoca era un giovane sacerdote e successivamente provinciale dei gesuiti, non condivideva molte delle posizioni espresse dai più radicali sostenitori della teologia della liberazione. Assunse quindi una posizione più moderata, sostenendo un impegno per la giustizia sociale, ma senza abbracciare i metodi e le idee rivoluzionarie associate al movimento. Sebbene il suo approccio fosse più sfumato e prudente rispetto a quello di alcuni suoi confratelli, sottolineò l'importanza dell'attenzione ai poveri e agli emarginati, pur rifiutando la violenza come strumento di cambiamento. Uno degli aspetti centrali del suo atteggiamento fu dunque il rifiuto della politicizzazione della religione.
'''

In [ ]:
tokens_long = long_text.lower().split()

len(tokens_long)

654

Build bigram counts on the larger dataset.

In [ ]:
bigram_counts_long = defaultdict(int)

for i in range(len(tokens_long)-1):
    bigram_counts_long[(tokens_long[i], tokens_long[i+1])] += 1

bigram_counts_long

defaultdict(int,
            {('nato', 'nel'): 1,
             ('nel', 'quartiere'): 1,
             ('quartiere', 'di'): 1,
             ('di', 'flores'): 1,
             ('flores', 'a'): 1,
             ('a', 'buenos'): 2,
             ('buenos', 'aires'): 3,
             ('aires', 'il'): 1,
             ('il', '17'): 1,
             ('17', 'dicembre'): 1,
             ('dicembre', '1936'): 1,
             ('1936', 'da'): 1,
             ('da', 'una'): 2,
             ('una', 'famiglia'): 1,
             ('famiglia', 'di'): 1,
             ('di', 'origini'): 1,
             ('origini', 'piemontesi'): 1,
             ('piemontesi', 'e'): 1,
             ('e', 'liguri'): 1,
             ('liguri', 'e'): 1,
             ('e', 'appartenente'): 1,
             ('appartenente', 'alla'): 1,
             ('alla', 'piccola'): 1,
             ('piccola', 'borghesia,'): 1,
             ('borghesia,', 'è'): 1,
             ('è', 'il'): 1,
             ('il', 'primogenito'): 1,
             ('pri

Let's inspect the **most common bigrams**.

In [ ]:
sorted(bigram_counts_long.items(), key=lambda x: x[1], reverse=True)[:10]

[(('e', 'il'), 4),
 (('teologia', 'della'), 4),
 (('buenos', 'aires'), 3),
 (('in', 'provincia'), 3),
 (('provincia', 'di'), 3),
 (('la', 'teologia'), 3),
 (('della', 'liberazione'), 3),
 (('a', 'buenos'), 2),
 (('da', 'una'), 2),
 (('da', 'parte'), 2)]

## Text Generation on the Larger Corpus

In [ ]:
context_counts_long = defaultdict(int)

for (w1, w2), c in bigram_counts_long.items():
    context_counts_long[w1] += c

bigram_probs_long = {}

for (w1, w2), c in bigram_counts_long.items():
    bigram_probs_long[(w1, w2)] = c / context_counts_long[w1]


def generate_bigram_long(start, length=20):

    word = start
    sent = [word]

    for _ in range(length):
        candidates = [(w2, p) for (w1, w2), p in bigram_probs_long.items() if w1 == word]

        if not candidates:
            break

        words, probs = zip(*candidates)
        word = random.choices(words, probs)[0]
        sent.append(word)

    return " ".join(sent)


generate_bigram_long("nato")

'nato in teologia presso il rapporto di cambiamento. uno degli oppressi, adottando una scuola tecnica argentina per completare il 1980 e'

# Exercises

1. Implement **Laplace smoothing for trigrams**
2. Compute **perplexity** of a sentence under the bigram model
3. Train the model on a **full book**
4. Compare **bigram vs trigram generation quality**